# 02 — Data Preprocessing & Zero-Leakage Pipeline
## Computer Engineering — Semester Machine Learning Project (Week 2)
### Key Engineering Rule:
To prevent **Data Leakage**, scalers and encoders must be fitted strictly on the **training set** (`X_train`) and only applied (`transform()`) to the test set (`X_test`).


In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

data_path = os.path.join("..", "Backend", "data", "Loan_default.csv")
if not os.path.exists(data_path):
    data_path = "Backend/data/Loan_default.csv"

df = pd.read_csv(data_path)
if "LoanID" in df.columns:
    df = df.drop(columns=["LoanID"])

X = df.drop(columns=["Default"])
y = df["Default"]

print(f"Features shape: {X.shape}, Target shape: {y.shape}")


## 1. Train-Test Split (BEFORE Any Scaling or Encoding)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"Train Default Rate: {y_train.mean():.4f}")
print(f"Test Default Rate:  {y_test.mean():.4f}")


## 2. Categorical Encoding (Fitted Only on X_train)


In [ ]:
categorical_cols = ["Education", "EmploymentType", "MaritalStatus", "HasMortgage", "HasDependents", "LoanPurpose", "HasCoSigner"]
encoders = {}

X_train_enc = X_train.copy()
X_test_enc = X_test.copy()

for col in categorical_cols:
    le = LabelEncoder()
    X_train_enc[col] = le.fit_transform(X_train[col].astype(str))
    X_test_enc[col] = le.transform(X_test[col].astype(str))
    encoders[col] = le
    print(f"Encoded {col}: {list(le.classes_)}")


## 3. Feature Scaling (StandardScaler Fitted Only on X_train)


In [ ]:
feature_order = ["Age", "Income", "LoanAmount", "CreditScore", "MonthsEmployed", "NumCreditLines", "InterestRate", "LoanTerm", "DTIRatio"] + categorical_cols

X_train_enc = X_train_enc[feature_order]
X_test_enc = X_test_enc[feature_order]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_enc)
X_test_scaled = scaler.transform(X_test_enc)

print(f"Scaled X_train mean: {X_train_scaled.mean():.4f}, std: {X_train_scaled.std():.4f}")
print(f"Scaled X_test mean:  {X_test_scaled.mean():.4f}, std: {X_test_scaled.std():.4f}")
print("Preprocessed feature space ready with ZERO data leakage.")
